In [1]:
# Basic Idea: 
# For GRPO training, we modify the MLP (up projection layers) from a Qwen model
# into an MoE module to train it with GRPO with multiple seeds
# (1). at first seed, we copy the current MLP weight into another MLP, creating an 
# MoE of 2 experts, then we "stiff route" to the first expert during GRPO training
# (2). at second seed, we "stiff route" to the second expert during GRPO training, 
#      .... (we "checkout S experts for a S seeds")
# I need this model wrapper first. 

# Training with MoE wrapper: 
# We'd add a second phase where we teach the model to "route", we freeze everything 
# but the router, train with (1). GRPO | (2). best of S experts + distillation loss (on router)



In [ ]:
# MoE-evolve wrapper. Full implementation: src/moe_evolve.py  (~90 lines)
import torch
from transformers import AutoModelForCausalLM
from src.moe_evolve import (
    convert_model_to_moe, train_expert, train_router, checkout_expert,
)

S = 3  # one expert per seed
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", torch_dtype=torch.bfloat16)
print("converted MLPs:", convert_model_to_moe(model, num_experts=S))

n_train = lambda: sum(p.numel() for p in model.parameters() if p.requires_grad)

# Phase 1: seed s trains only expert s (hard-routed). Plug GRPOTrainer in the loop.
for s in range(S):
    train_expert(model, s)
    print(f"seed {s}: hard-route expert {s}, trainable = {n_train():,}")

# Phase 2: freeze experts, train only the routers (soft per-token mix).
train_router(model)
print(f"router phase: trainable = {n_train():,}")

# this is a clean kernel, love it. now we need a "moepo_game24.py" that implements the moe + grpo training
# (1). for the router training phase, we need to use each expert to carry out one rollout (greedy rollout)
#      then assign advantage per GRPO style (except that in the group, we now have rollouts from different experts)
#      we train the router's routing logits, so as to prefer the experts that produce the better greedy rollout
# (2). we can simply the script, since I don't really need all those fancy tricks / levers that TreeTrainer maintains
#      even better, we can checkout a separate trainer, inheriting from GRPO trainer and call it 'MoPOTrainer' instead
#      to carry out such multi-seed + router training pipeline 

# checkout_expert(model, src, dst)  # optional: seed a new expert from an old one


`torch_dtype` is deprecated! Use `dtype` instead!


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

converted MLPs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
seed 0: hard-route expert 0, trainable = 264,241,152
seed 1: hard-route expert 1, trainable = 264,241,152
seed 2: hard-route expert 2, trainable = 264,241,152
router phase: trainable = 86,016


The biggest bet here, is that by training the model to "route" to different experts (each trained via different seeds of GRPO)
model can itself learns to recover from the "best" expert for each query. 
[Issue]. However, whilst we'd like the routing decision to be made ONCE at the end of query, GRPO trains on each expert
         so that they already process each query differently, but it's not possible to just route to the "right expert" 
         from the start of query --- MoE are fundamentally a per-token routing mechanism. 
[Dumb Idea]. Via the "rollout from all experts" -> form a group -> calculate advantage -> scale router gradient pipeline
             we can supervise the routing decision on EACH TOKEN to align with the best expert via best of all enumeration selection. This way we always ensure the learned routing decision mimic the "optimal per-expert performance"
             initial prefix routing will be noisy, and this is the main gap of this idea, how do we overcome this issue?


Phase I. (Multi-seed training)
- for seed s, train expert s (hard route), freeze router, train expert
- for query, use expert 0, for prompt, use expert s
Phase 2. (Seed ensemble training)
- for query, use expert 0, train router on end-of-query token's
  representation to route towards "best expert" (which is selected 
  from group of S rollouts, each from a greedy-response from an expert)
[Optionally] expert 0 can be trainable or not. 
[Prior Experiment]
- verify that different seed in Phase I have different behavior (union acc > avg. acc) suffices for this validation

[Critique]. well that makes the gadget very ugly and complicated (code wise)

How about this, in Phase I

S = 4 (4 experts, 4 seeds)
[Phase I. Multi-seed training]
- for seed s, train expert s (hard route), freeze router, train expert
- same routing pattern for prompt and response, all use expert s
[Phase II. Seed ensemble training]
- group consists of 8 rollouts consisting of 4 expert (hard routed) greedy response, as well as 4 self-routed rollout (top-1 routing) response, GRPO
style adantage scaled routing loss (on all tokens, at all layers' MoE routers) Expert frozen, train only the router
- note that the per-expert greedy rollout can be obtained ONCE before Phase II since we froze expert in Phase II anyway, we need only online rollout from the 4 self-routed ver.
[Evaluation]
- top-1 MoE routing based evaluation

[On Distillation]
- this is a valid methodology, with the MoE tricks working, we'd eventually 
  want to achieve distillation, too

In [ ]:
# MoPO: multi-seed expert GRPO + router training. Trainer: src/mopo_trainer.py (~190 lines)
# Both phases are one REINFORCE update; only the "policy" differs:
#   phase 1 -> token policy of expert s (hard-routed, sampled G/prompt)
#   phase 2 -> router's categorical over experts (each expert: 1 greedy rollout/prompt -> group of S)
# vLLM can't run custom routing, so generation is plain HF generate (honours MoE mode).
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.moe_evolve import convert_model_to_moe
from src.mopo_trainer import MoPOTrainer
from src.game24utils import build_puzzle_pool, split_train_eval, make_dataset, correctness_reward, format_reward

import random; random.seed(0); torch.manual_seed(0)
train_ds = make_dataset(split_train_eval(build_puzzle_pool(max_n=9), rng=random.Random(0xE7A15))[0])

S = 3
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", dtype=torch.bfloat16).to("cuda")
convert_model_to_moe(model, num_experts=S)

tr = MoPOTrainer(model, tok, [correctness_reward, format_reward],
                 num_experts=S, lr=5e-6, max_new_tokens=512, group_size=8)
tr.train_experts(train_ds, steps_per_expert=30, temperature=1.0, batch_prompts=4)  # phase 1
tr.train_router(train_ds, steps=30, batch_prompts=4)                               # phase 2

# CLI equivalent: python script/moepo_game24.py --experts 3 --expert-steps 30 --router-steps 30
